# Illustration of Splines
We now provide a piece of code that takes advantage of the ``splinekit`` library to build and display random polynomial regular splines characterized by their degree, which the user can select.

The displayed curve carries two markers.
*   The hollow circles correspond to the samples of the spline, at the integer arguments. Stems extend from the sample values to the zero baseline.
*   The smaller (black) dots indicate those locations of the spline curve where its high-order derivatives cease to exist. These dots coincide with the integer samples when the degree is odd; they are off by one half when the degree is even.

Visually, the splines of degree zero and the linear splines have a distinctive appearance. Meanwhile, the general appearance of quadratic splines is very difficult to tell apart from that of splines of higher degrees. In reality, however, the smoothness of the splines always increases with their degree, and so does their order of approximation.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
min_degree = 0 # Minimal spline degree
max_degree = 9 # Maximal spline degree
support = 6 # Length of the displayed spline
step_rate = 8 # Number of steps per unit domain
hertz = 12 # Refresh rate
pts = 2 * step_rate * support + 1 # Overall number of plot samples

t = 0 # Time
s = sk.PeriodicSpline1D.from_spline_coeff(
    np.random.standard_normal(support + max_degree + 2), # Initial coefficients
    degree = 3 # Initial degree
) # Initial spline

# Layout of the plot
plotrange = sk.interval.Closed((-2, 2))
hdisplay = display("", display_id = True)

# Display one frame
def plot_spline (
    t
):
    p = 2 * t - (s.degree + 1) * step_rate
    q = 2 * step_rate
    if 0 == p % q:
        # Innovation, create a new coefficient outside of the current plot
        s.spline_coeff[(p // q) % s.period] = np.random.standard_normal()
    (fig, ax) = plt.subplots()
    s.plot(
        (fig, ax),
        plotpoints = pts,
        plotdomain = sk.interval.ClosedOpen((
            (0.5 + t) / step_rate,
            (0.5 + t) / step_rate + support
        )), # Running domain
        plotrange = plotrange,
        periodbound_markerfmt = "oC0",
        periodboundstem_linefmt = "-C0"
    )
    plt.close()
    hdisplay.update(fig)

# Allow for the selection of the degree
degree_slider = widgets.IntSlider(min = min_degree, max = max_degree, value = s.degree)
def on_degree_changed (
    change
):
    global s
    samples = s.get_samples(0.0, support_length = s.period)
    s = sk.PeriodicSpline1D.from_samples(samples, degree = change["new"])
    plot_spline()
degree_slider.observe(on_degree_changed, names = "value")

# Stepper
stepper_widget = widgets.Play(
    value = 0,
    min = 0,
    max = step_rate - 1,
    step = 1,
    interval = 1000 / hertz,
    disabled = False,
    repeat = True,
    show_repeat = False
)
def on_did_step (
    change
):
    global t
    plot_spline(t)
    t += 1
stepper_widget.observe(on_did_step, names = "value")

# Show controls
widgets.VBox([
    widgets.HBox([widgets.Label(value='Degree:'), degree_slider]),
    widgets.HBox([widgets.Label(value='Play/Pause:'), stepper_widget])
])
